# Iceberg V3 Basics

This notebook demonstrates creating Iceberg v3 tables with Snowflake managed storage, including:
- VARIANT columns for semi-structured data
- Nanosecond precision timestamps
- Basic DML operations

In [ ]:
-- Set context
USE ROLE ACCOUNTADMIN;
USE DATABASE ICEBERG_TESTING;
USE SCHEMA PUBLIC;
USE WAREHOUSE COMPUTE_WH;

In [ ]:
-- Create Iceberg v3 table with VARIANT and nanosecond timestamp
CREATE OR REPLACE ICEBERG TABLE CUSTOMER_EVENTS_V3 (
    event_id INT,
    customer_id INT,
    event_type STRING,
    event_timestamp TIMESTAMP_NTZ(9),
    event_data VARIANT,
    event_metadata VARIANT
)
CATALOG = 'SNOWFLAKE'
EXTERNAL_VOLUME = 'EXVOL'
ICEBERG_VERSION = 3;

In [ ]:
-- Insert sample data with nanosecond precision timestamps and nested JSON
INSERT INTO CUSTOMER_EVENTS_V3 
SELECT 
    1, 101, 'login', 
    '2026-03-11 10:00:00.123456789'::TIMESTAMP_NTZ(9),
    PARSE_JSON('{"device": "mobile", "os": "iOS", "version": "17.2"}'),
    PARSE_JSON('{"session_id": "abc123", "ip": "192.168.1.1"}')
UNION ALL SELECT 
    2, 101, 'purchase',
    '2026-03-11 10:15:00.987654321'::TIMESTAMP_NTZ(9),
    PARSE_JSON('{"amount": 99.99, "item": "widget", "qty": 2}'),
    PARSE_JSON('{"payment_method": "credit", "currency": "USD"}')
UNION ALL SELECT
    3, 102, 'login',
    '2026-03-11 10:30:00.111222333'::TIMESTAMP_NTZ(9),
    PARSE_JSON('{"device": "desktop", "browser": "Chrome"}'),
    PARSE_JSON('{"session_id": "xyz789", "geo": {"country": "US", "state": "CA"}}');

In [ ]:
-- Query the table, showing nanosecond precision
SELECT 
    event_id, 
    event_type, 
    TO_VARCHAR(event_timestamp, 'YYYY-MM-DD HH24:MI:SS.FF9') AS timestamp_ns,
    event_data,
    event_metadata
FROM CUSTOMER_EVENTS_V3
ORDER BY event_id;

In [ ]:
-- Query nested VARIANT data
SELECT 
    event_id,
    event_data:device::STRING AS device,
    event_metadata:geo:country::STRING AS country,
    event_metadata:geo:state::STRING AS state
FROM CUSTOMER_EVENTS_V3
WHERE event_metadata:geo IS NOT NULL;

In [ ]:
-- Show Iceberg table metadata
SHOW ICEBERG TABLES LIKE 'CUSTOMER_EVENTS_V3';

## Key Takeaways

- **No BASE_LOCATION required** - Snowflake auto-generates the path
- **ICEBERG_VERSION = 3** enables nanosecond timestamps and VARIANT support
- **VARIANT columns** work seamlessly with standard Snowflake JSON notation
- **Full DML support** - INSERT, UPDATE, DELETE, MERGE all work